# Airbnb Cape Town Capstone (Starter Notebook)

**Student:** Precious  
**Dataset:** Inside Airbnb, Cape Town listings snapshot (2026-09-23), CC BY 4.0  

**Two tasks, one dataset (clean/EDA/split shared):**
- **Task C (classification, deployed):** predict `host_is_superhost`
- **Task R (regression, notebook only):** predict `price`

> This is a **starter/sample notebook**. You must understand, edit, and be able to explain every cell out loud.  
> Replace all `TODO` items with your own work and add markdown findings after important results.

## A–B. Setup, framing, and targets (Checklist A, B)

**TODO (markdown, in your own words):**
- Who uses each prediction and what decision it supports?
- Why is Task C classification and Task R regression?
- Define targets: `host_is_superhost` → 0/1 (print class counts). `price` → numeric (state what you do with missing/invalid prices).
- Choose metrics **before** modelling and justify them (Task C: why not accuracy alone; Task R: RMSE vs MAE).

In [1]:
import sys, subprocess
print(sys.executable)
subprocess.check_call([sys.executable, "-m", "pip", "show", "xgboost"])

c:\Users\Chibi\Desktop\New folder\Workspace\Project\CAMLDS\Final\venv\Scripts\python.exe


0

In [2]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import (
    GroupShuffleSplit, GroupKFold, cross_validate, cross_val_predict,
    RandomizedSearchCV
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay,
    mean_squared_error, mean_absolute_error, silhouette_score
)
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception as e:
    XGBOOST_AVAILABLE = False
    print('XGBoost not available:', e)

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 140)
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Everything works')
print('pandas', pd.__version__)
print('numpy', np.__version__)
import sklearn
print('sklearn', sklearn.__version__)
print('xgboost available:', XGBOOST_AVAILABLE)

Everything works
pandas 2.3.3
numpy 2.0.2
sklearn 1.6.1
xgboost available: True


In [ ]:
# Paths (brief layout: data/raw for original files, data/processed for cleaned outputs)
PROJECT_DIR = Path('..').resolve()  # if notebook is in notebooks/
DATA_DIR = PROJECT_DIR / 'data' / 'raw'
LISTINGS_PATH = DATA_DIR / 'listings.csv.gz'
NEIGH_PATH = DATA_DIR / 'neighbourhoods.csv'
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
MODELS_DIR = PROJECT_DIR / 'models'
FIG_DIR = PROJECT_DIR / 'reports' / 'figures'
for p in [PROCESSED_DIR, MODELS_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('Listings exists:', LISTINGS_PATH.exists(), LISTINGS_PATH)
print('Neighbourhoods exists:', NEIGH_PATH.exists(), NEIGH_PATH)

## C. Data loading and inspection (Checklist C)

Tasks: load raw gz, print shape (expect ~27760 x 90), inspect dtypes/missing/value counts, fix `price` and `bathrooms_text`, audit missing/empty cols, duplicates and host repeats, merge `neighbourhoods.csv` and verify row counts.

In [ ]:
# C1. Load
df_raw = pd.read_csv(LISTINGS_PATH, compression='gzip', low_memory=False)
print(df_raw.shape)
df_raw.head()

In [ ]:
# C2. Inspect
df_raw.info()
# df_raw.describe(include='all').T
# TODO: value_counts for key categoricals; write a markdown summary of what each column type seems to be

In [ ]:
# C3. Dtype fixes (examples/starter - verify and adapt)
df = df_raw.copy()

# price: '$1,114.00' -> float (and blanks)
df['price_clean'] = (
    df['price'].astype(str)
      .str.replace('$', '', regex=False)
      .str.replace(',', '', regex=False)
      .replace({'': np.nan, 'nan': np.nan})
)
df['price_clean'] = pd.to_numeric(df['price_clean'], errors='coerce')

# bathrooms_text: extract numeric count (handle 'half-bath' cases carefully)
# TODO: improve parsing and show before/after examples
df['bathrooms_count'] = df['bathrooms_text'].str.extract(r'(\d+\.?\d*)').astype(float)

print(df[['price', 'price_clean']].head(10))
print(df[['bathrooms_text', 'bathrooms_count']].head(10))

In [ ]:
# C4. Missing audit
missing = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_pct': df.isna().mean() * 100
}).sort_values('missing_pct', ascending=False)
missing.head(30)
# TODO: identify fully-empty columns in this snapshot; document per-column decisions

In [ ]:
# C5. Duplicates and host repeats
print('Duplicate rows:', df.duplicated().sum())
host_counts = df['host_id'].value_counts()
print('Hosts with >1 listing:', (host_counts > 1).sum())
print(host_counts.head())
# TODO: markdown - what this means for splitting and host-level features

In [ ]:
# C6. Merge neighbourhoods
neigh = pd.read_csv(NEIGH_PATH)
print(neigh.head(), neigh.shape)
# TODO: choose join key/type (e.g. neighbourhood_cleansed vs neighbourhood), check row counts before/after
# df_merged = df.merge(neigh, left_on='neighbourhood_cleansed', right_on='neighbourhood', how='left')
# print(df.shape, df_merged.shape)

## D. Cleaning (Checklist D)

Do cleaning in code, in order, reproducibly. Remove useless columns with reasons, handle rows/prices/min nights with justification, examine price skew/transform, and create at least one engineered feature (e.g. amenities count) to evaluate later (I5).

In [ ]:
# D1–D3 starter skeleton (adapt heavily)
# drop_cols = [...]
# df_clean = df_merged.drop(columns=[c for c in drop_cols if c in df_merged.columns]).copy()
# TODO: document each group of dropped columns/rows
# TODO: save processed copy
# df_clean.to_parquet(PROCESSED_DIR / 'listings_clean.parquet', index=False)  # or csv

In [ ]:
# D4. Price distribution and transform (example)
# fig, axes = plt.subplots(1, 2, figsize=(12,4))
# df_clean['price_clean'].dropna().clip(upper=df_clean['price_clean'].quantile(0.99)).hist(bins=50, ax=axes[0])
# axes[0].set_title('Price (clipped for view)')
# np.log1p(df_clean['price_clean'].dropna()).hist(bins=50, ax=axes[1])
# axes[1].set_title('log1p(price)')
# plt.show()
# TODO: decide and explain; show inverse transform back to rand if used

In [ ]:
# D5. Engineered feature example: amenities count (TODO: make robust)
# df_clean['amenities_count'] = df_clean['amenities'].fillna('[]').astype(str).str.count(',') + 1
# df_clean.loc[df_clean['amenities'].fillna('[]').astype(str).str.strip().isin(['[]','', 'nan']), 'amenities_count'] = 0
# df_clean['amenities_count'].describe()

## E. EDA with written findings (Checklist E)

Include: target plots, ≥4 distributions, ≥4 feature-vs-target plots, at least one `groupby().agg()` table, correlation check, and a **5-point EDA narrative** in your own words (each point tied to evidence).

In [ ]:
# E1–E6 placeholders
# TODO: create plots/tables and add markdown findings immediately after each

## F. Leakage review and split (Checklist F)

**Critical:** write leakage review before modelling (include Airbnb Superhost rules in your own summary).  
Use a **group-aware split** so all listings from the same `host_id` stay in the same set; compare with a plain random split on one model to show why it matters.  
Create train/val/test with fixed seed and report sizes/class rates. Fit preprocessing on train only.

In [ ]:
# Target prep (example)
# df_model = df_clean.copy()
# df_model['is_superhost'] = df_model['host_is_superhost'].map({'t': 1, 'f': 0})
# TODO: handle missing target/price rows per your decisions
# TODO: define feature sets for Task C and Task R (after leakage review)

# Group-aware split example
# gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_STATE)
# train_idx, temp_idx = next(gss.split(df_model, groups=df_model['host_id']))
# train_df = df_model.iloc[train_idx]
# temp_df = df_model.iloc[temp_idx]
# gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=RANDOM_STATE)
# val_idx, test_idx = next(gss2.split(temp_df, groups=temp_df['host_id']))
# val_df = temp_df.iloc[val_idx]
# test_df = temp_df.iloc[test_idx]
# print(len(train_df), len(val_df), len(test_df))
# print(train_df['is_superhost'].mean(), val_df['is_superhost'].mean(), test_df['is_superhost'].mean())

## G. Baselines (Checklist G)

- Task C: `DummyClassifier` (majority) with accuracy/precision/recall/F1 on validation.
- Task R: mean/median predictor with RMSE/MAE on validation.
- One plain-language bias/variance sentence tied to your results (update as you compare models in H).

In [ ]:
# TODO: implement baselines for both tasks

## H–J. Modelling, workflow, evaluation (Checklist H, I, J)

Build pipelines (ColumnTransformer for numeric/categorical), try required models, use GroupKFold CV and tuning, handle imbalance, evaluate once on test, and perform error/subgroup analysis.  
Keep the **test set untouched** until final scoring (J1).

In [ ]:
# Starter: define feature lists and a reusable preprocessor (edit to your actual columns)
# num_features = [...]
# cat_features = [...]
# numeric_transformer = Pipeline(steps=[
#     ('imputer', SimpleImputer(strategy='median')),
#     ('scaler', StandardScaler())
# ])
# categorical_transformer = Pipeline(steps=[
#     ('imputer', SimpleImputer(strategy='most_frequent')),
#     ('onehot', OneHotEncoder(handle_unknown='ignore'))
# ])
# preprocess = ColumnTransformer([
#     ('num', numeric_transformer, num_features),
#     ('cat', categorical_transformer, cat_features)
# ])

# Example model pipeline
# clf_logreg = Pipeline([('preprocess', preprocess), ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))])
# TODO: fit, validate, compare, tune, and explain

## K. Unsupervised piece (Checklist K)

K-means on scaled numeric features (elbow + silhouette, interpret centroids), PCA 2D plot colored by cluster with explained variance, and connect clusters to Superhost rate / median price.

In [ ]:
# TODO: clustering and PCA

## L. Save Superhost pipeline for the app (Checklist L)

- Train the **final deployed Superhost pipeline on the 8–12 app features only** (same columns/types the app will send).
- Save the whole pipeline (preprocessing + model) with `joblib`.
- Reload in a fresh cell and verify identical prediction on a sample row. Check file size (<100 MB).

In [ ]:
# TODO: define APP_FEATURES, train final pipeline on them, save/reload/verify
# joblib.dump(superhost_app_pipeline, MODELS_DIR / 'superhost_pipeline.joblib')
# loaded = joblib.load(MODELS_DIR / 'superhost_pipeline.joblib')

## O (ongoing). Decision log and findings

Keep a running decision log (for README, ≥4 key entries): **decision | options considered | why chosen**.  
Examples: leakage columns removed, split strategy, target/price handling, app feature selection, threshold choice, model tradeoffs.

| Decision | Options considered | Why chosen |
|---|---|---|
| TODO | TODO | TODO |
| TODO | TODO | TODO |
| TODO | TODO | TODO |
| TODO | TODO | TODO |

## Final checks before submission/presentation
- [ ] Kernel restarted and **Run All** completes with no errors/manual edits (O2)
- [ ] Every important result has a written finding in your own words (O3)
- [ ] Test set scored once per task (J1)
- [ ] `models/superhost_pipeline.joblib` loads and matches original predictions (L2)
- [ ] `app.py` runs locally and handles inputs correctly (M2, M4)
- [ ] README includes live URL, credit, 5 findings, scores, limits, run steps, decision log, next steps (O1)
- [ ] You can explain and live-edit random cells (A1 spirit / O5)